# 01 - Testar observabilidade dos exutórios

Este notebook faz um censo preliminar de observabilidade SWOT para os 13 exutórios. A etapa consulta apenas metadados NASA Earthdata/PO.DAAC via `earthaccess` e não baixa arquivos PIXC, RiverSP ou LakeSP.

## Objetivo

Verificar quais produtos/passagens SWOT aparecem no catálogo para a área que envolve os exutórios, gerando uma tabela simples de cobertura potencial por ponto. O resultado não substitui a análise pixel a pixel: ele apenas indica se há metadados de passagens/produtos interceptando a área de interesse.

## Dados de entrada

Entrada esperada: `dados/exutorios.csv`, com exatamente 13 linhas e as colunas `id`, `latitude`, `longitude`. No ambiente local atual, o notebook também aceita o arquivo em `../dados/exutorios.csv`, sem alterá-lo.

In [1]:
from __future__ import annotations

import json
import logging
import re
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import geopandas as gpd
from shapely.geometry import box


In [2]:
def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'src' / 'check_environment.py').exists():
            return candidate
    fallback = Path.home() / 'mystorage' / 'PPGGAG1889' / 'atividade3_swot'
    if fallback.exists():
        return fallback
    raise RuntimeError('FALHA: rode este notebook dentro do repositorio atividade3_swot.')

PROJECT_ROOT = find_project_root(Path.cwd())
INPUT_CANDIDATES = [
    PROJECT_ROOT / 'dados' / 'exutorios.csv',
    PROJECT_ROOT.parent / 'dados' / 'exutorios.csv',
]
OUTPUT_TABLE = PROJECT_ROOT / 'outputs' / 'tabelas' / 'observabilidade_exutorios.csv'
LOG_FILE = PROJECT_ROOT / 'outputs' / 'logs' / '01_testar_observabilidade_pontos.log'

OUTPUT_TABLE.parent.mkdir(parents=True, exist_ok=True)
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    filename=LOG_FILE,
    filemode='w',
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
)

print('OK raiz do projeto:', PROJECT_ROOT)
print('OK log:', LOG_FILE)


OK raiz do projeto: /home/jovyan/mystorage/PPGGAG1889/atividade3_swot
OK log: /home/jovyan/mystorage/PPGGAG1889/atividade3_swot/outputs/logs/01_testar_observabilidade_pontos.log


In [3]:
def resolve_input_csv() -> Path:
    for path in INPUT_CANDIDATES:
        if path.exists():
            return path
    candidates = '\n'.join(str(p) for p in INPUT_CANDIDATES)
    raise FileNotFoundError(f'FALHA: exutorios.csv nao encontrado. Caminhos testados:\n{candidates}')

input_csv = resolve_input_csv()
df = pd.read_csv(input_csv)

expected_columns = ['id', 'latitude', 'longitude']
if list(df.columns) != expected_columns:
    raise ValueError(f'FALHA: colunas esperadas {expected_columns}, colunas encontradas {list(df.columns)}')
if len(df) != 13:
    raise ValueError(f'FALHA: esperados 13 exutorios, encontrados {len(df)}')
if df['id'].isna().any() or df['id'].duplicated().any():
    raise ValueError('FALHA: coluna id contem nulos ou duplicados.')

df['latitude'] = pd.to_numeric(df['latitude'], errors='raise')
df['longitude'] = pd.to_numeric(df['longitude'], errors='raise')
if not df['latitude'].between(-90, 90).all():
    raise ValueError('FALHA: latitude fora do intervalo [-90, 90].')
if not df['longitude'].between(-180, 180).all():
    raise ValueError('FALHA: longitude fora do intervalo [-180, 180].')

logging.info('CSV lido com sucesso: %s', input_csv)
logging.info('Total de exutorios: %s', len(df))
print('OK entrada:', input_csv)
display(df)


OK entrada: /home/jovyan/mystorage/PPGGAG1889/atividade3_swot/dados/exutorios.csv


,id,latitude,longitude
0,P01,-15.539765,-47.972069
1,P02,-15.541334,-47.969589
2,P03,-15.541921,-47.981644
3,P04,-15.542892,-47.980763
4,P05,-15.542658,-47.979778
5,P06,-15.543083,-47.979286
6,P07,-15.541673,-47.979366
7,P08,-15.537334,-47.975750
8,P09,-15.537017,-47.975281
9,P10,-15.536032,-47.976860


## Metodologia

1. Converter os exutórios em pontos geográficos (`EPSG:4326`).
2. Calcular o envelope espacial do conjunto.
3. Reprojetar para um CRS métrico local, aplicar buffer pequeno de segurança e voltar para `EPSG:4326`.
4. Consultar o catálogo NASA Earthdata/CMR via `earthaccess.search_data`, sem download.
5. Resumir as passagens/produtos encontrados e gravar uma tabela por ponto.

In [4]:
BUFFER_METERS = 5000

gdf = gpd.GeoDataFrame(
    df.copy(),
    geometry=gpd.points_from_xy(df['longitude'], df['latitude']),
    crs='EPSG:4326',
)

utm_crs = gdf.estimate_utm_crs()
if utm_crs is None:
    raise RuntimeError('FALHA: nao foi possivel estimar CRS metrico para o buffer.')

gdf_m = gdf.to_crs(utm_crs)
buffered_union = gdf_m.geometry.buffer(BUFFER_METERS).union_all()
bbox_wgs84 = gpd.GeoSeries([buffered_union.envelope], crs=utm_crs).to_crs('EPSG:4326').iloc[0]
minx, miny, maxx, maxy = bbox_wgs84.bounds
bbox = (minx, miny, maxx, maxy)

logging.info('CRS metrico estimado: %s', utm_crs)
logging.info('Buffer aplicado: %s metros', BUFFER_METERS)
logging.info('Bounding box WGS84: %s', bbox)
print('OK CRS metrico:', utm_crs)
print('OK buffer_m:', BUFFER_METERS)
print('OK bounding_box:', bbox)
display(gdf)


OK CRS metrico: EPSG:32723
OK buffer_m: 5000
OK bounding_box: (-48.02893635512781, -15.59256934663154, -47.915784197830234, -15.490193219201679)


,id,latitude,longitude,geometry
0,P01,-15.539765,-47.972069,POINT (-47.97207 -15.53976)
1,P02,-15.541334,-47.969589,POINT (-47.96959 -15.54133)
2,P03,-15.541921,-47.981644,POINT (-47.98164 -15.54192)
3,P04,-15.542892,-47.980763,POINT (-47.98076 -15.54289)
4,P05,-15.542658,-47.979778,POINT (-47.97978 -15.54266)
5,P06,-15.543083,-47.979286,POINT (-47.97929 -15.54308)
6,P07,-15.541673,-47.979366,POINT (-47.97937 -15.54167)
7,P08,-15.537334,-47.975750,POINT (-47.97575 -15.53733)
8,P09,-15.537017,-47.975281,POINT (-47.97528 -15.53702)
9,P10,-15.536032,-47.976860,POINT (-47.97686 -15.53603)


## Consulta de metadados SWOT

A célula abaixo consulta apenas o catálogo. Ela usa `earthaccess.search_data` com `bounding_box` e não chama `earthaccess.download`. Se o ambiente exigir autenticação Earthdata, o erro será preservado no output e registrado no log.

In [8]:
try:
    import earthaccess
except Exception as exc:
    logging.exception('Falha ao importar earthaccess')
    raise RuntimeError('FALHA: earthaccess nao esta instalado. Rode pip install -r requirements.txt.') from exc

START_DATE = '2023-01-01'
END_DATE = datetime.now(timezone.utc).date().isoformat()
PRODUCTS = {
    'SWOT_L2_HR_RIVERSP_D': '*Reach*',
    'SWOT_L2_HR_LAKESP_D': '*Prior*',
    'SWOT_L2_HR_PIXC_D': '*',
}

def granule_umm(granule):
    if hasattr(granule, 'umm'):
        return granule.umm
    if isinstance(granule, dict):
        return granule.get('umm', granule)
    try:
        return granule['umm']
    except Exception:
        return {}

def granule_native_id(granule) -> str:
    umm = granule_umm(granule)
    return (
        umm.get('GranuleUR')
        or umm.get('ProducerGranuleId')
        or str(getattr(granule, 'data_links', lambda: [])())
        or str(granule)
    )

def granule_date(umm: dict, name: str):
    temporal = umm.get('TemporalExtent', {}) or {}
    range_time = temporal.get('RangeDateTime', {}) or {}
    value = range_time.get('BeginningDateTime')
    if value:
        return pd.to_datetime(value, errors='coerce')
    match = re.search(r'_(20\d{6})T', name)
    if match:
        return pd.to_datetime(match.group(1), format='%Y%m%d', errors='coerce')
    return pd.NaT

def parse_cycle_pass_tile(name: str) -> dict:
    match = re.search(r'SWOT_L2_HR_(?:PIXC|RiverSP_Reach|LakeSP_Prior)_(\d{3})_(\d{3})(?:_([0-9A-Z]+))?', name, re.IGNORECASE)
    if not match:
        return {'cycle': None, 'pass': None, 'tile': None}
    return {'cycle': match.group(1), 'pass': match.group(2), 'tile': match.group(3)}

records = []
for short_name, granule_pattern in PRODUCTS.items():
    logging.info('Consultando %s', short_name)
    try:
        results = earthaccess.search_data(
            short_name=short_name,
            temporal=(START_DATE, END_DATE),
            bounding_box=bbox,
            granule_name=granule_pattern,
            count=-1,
        )
    except Exception as exc:
        logging.exception('Falha na consulta %s', short_name)
        raise RuntimeError(
            f'FALHA na consulta Earthdata/CMR para {short_name}. '
            'Verifique internet, pacote earthaccess e, se necessario, login Earthdata manual.'
        ) from exc

    logging.info('%s: %s granulos encontrados', short_name, len(results))
    print(f'OK consulta {short_name}: {len(results)} granulos')
    for granule in results:
        umm = granule_umm(granule)
        native_id = granule_native_id(granule)
        parsed = parse_cycle_pass_tile(native_id)
        records.append({
            'short_name': short_name,
            'granule_id': native_id,
            'data': granule_date(umm, native_id),
            'cycle': parsed['cycle'],
            'pass': parsed['pass'],
            'tile': parsed['tile'],
        })

metadata = pd.DataFrame.from_records(records)
if not metadata.empty:
    metadata = metadata.sort_values(['data', 'short_name', 'granule_id'], na_position='last').reset_index(drop=True)

print('Total de metadados encontrados:', len(metadata))
display(metadata.head(20))


/opt/conda/lib/python3.11/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


OK consulta SWOT_L2_HR_RIVERSP_D: 472 granulos
OK consulta SWOT_L2_HR_LAKESP_D: 478 granulos
OK consulta SWOT_L2_HR_PIXC_D: 216 granulos
Total de metadados encontrados: 1166


,short_name,granule_id,data,cycle,pass,tile
0,SWOT_L2_HR_RIVERSP_D,SWOT_L2_HR_RiverSP_Reach_001_158_SA_20230726T2...,2023-07-26 20:35:03.190000+00:00,001,158,SA
1,SWOT_L2_HR_LAKESP_D,SWOT_L2_HR_LakeSP_Prior_001_158_SA_20230726T20...,2023-07-26 20:35:03.195000+00:00,001,158,SA
2,SWOT_L2_HR_RIVERSP_D,SWOT_L2_HR_RiverSP_Reach_001_227_SA_20230729T0...,2023-07-29 07:37:28.909000+00:00,001,227,SA
3,SWOT_L2_HR_LAKESP_D,SWOT_L2_HR_LakeSP_Prior_001_227_SA_20230729T07...,2023-07-29 07:37:28.915000+00:00,001,227,SA
4,SWOT_L2_HR_LAKESP_D,SWOT_L2_HR_LakeSP_Prior_001_227_SA_20230729T07...,2023-07-29 07:37:33.808000+00:00,001,227,SA
5,SWOT_L2_HR_PIXC_D,SWOT_L2_HR_PIXC_001_227_127L_20230729T074154_2...,2023-07-29 07:41:54.101000+00:00,001,227,127L
6,SWOT_L2_HR_RIVERSP_D,SWOT_L2_HR_RiverSP_Reach_001_255_SA_20230730T0...,2023-07-30 07:36:58.828000+00:00,001,255,SA
7,SWOT_L2_HR_LAKESP_D,SWOT_L2_HR_LakeSP_Prior_001_255_SA_20230730T07...,2023-07-30 07:37:04.878000+00:00,001,255,SA
8,SWOT_L2_HR_RIVERSP_D,SWOT_L2_HR_RiverSP_Reach_001_408_SA_20230804T1...,2023-08-04 18:58:25.883000+00:00,001,408,SA
9,SWOT_L2_HR_LAKESP_D,SWOT_L2_HR_LakeSP_Prior_001_408_SA_20230804T18...,2023-08-04 18:58:25.888000+00:00,001,408,SA


In [9]:
if metadata.empty:
    n_passagens = 0
    primeira_data = pd.NaT
    ultima_data = pd.NaT
    cobertura_potencial = 'nao_identificada'
    observacoes = 'Nenhum metadado SWOT encontrado para o envelope com buffer nesta consulta.'
else:
    # Nesta etapa preliminar, a cobertura e avaliada pelo envelope do conjunto.
    # A interseccao exata por ponto exige leitura de footprint/granulo ou produto bruto, etapa posterior.
    unique_passages = metadata[['short_name', 'cycle', 'pass', 'tile', 'granule_id']].drop_duplicates()
    n_passagens = len(unique_passages)
    primeira_data = metadata['data'].dropna().min()
    ultima_data = metadata['data'].dropna().max()
    cobertura_potencial = 'sim' if n_passagens > 0 else 'nao_identificada'
    products = ', '.join(sorted(metadata['short_name'].dropna().unique()))
    observacoes = (
        'Cobertura potencial inferida por metadados no envelope com buffer; '
        f'produtos consultados/encontrados: {products}. Validar interseccao fina em etapa posterior.'
    )

out = df.rename(columns={'latitude': 'ponto_latitude', 'longitude': 'ponto_longitude'}).copy()
out['n_passagens_encontradas'] = n_passagens
out['primeira_data'] = '' if pd.isna(primeira_data) else pd.to_datetime(primeira_data).date().isoformat()
out['ultima_data'] = '' if pd.isna(ultima_data) else pd.to_datetime(ultima_data).date().isoformat()
out['cobertura_potencial'] = cobertura_potencial
out['observacoes'] = observacoes

columns = [
    'id', 'ponto_latitude', 'ponto_longitude',
    'n_passagens_encontradas', 'primeira_data', 'ultima_data',
    'cobertura_potencial', 'observacoes'
]
out = out[columns]
out.to_csv(OUTPUT_TABLE, index=False, encoding='utf-8')
logging.info('Tabela de observabilidade salva em %s', OUTPUT_TABLE)
print('OK tabela salva:', OUTPUT_TABLE)
display(out)


OK tabela salva: /home/jovyan/mystorage/PPGGAG1889/atividade3_swot/outputs/tabelas/observabilidade_exutorios.csv


,id,ponto_latitude,ponto_longitude,n_passagens_encontradas,primeira_data,ultima_data,cobertura_potencial,observacoes
0,P01,-15.539765,-47.972069,1166,2023-07-26,2026-08-30,sim,Cobertura potencial inferida por metadados no ...
1,P02,-15.541334,-47.969589,1166,2023-07-26,2026-08-30,sim,Cobertura potencial inferida por metadados no ...
2,P03,-15.541921,-47.981644,1166,2023-07-26,2026-08-30,sim,Cobertura potencial inferida por metadados no ...
3,P04,-15.542892,-47.980763,1166,2023-07-26,2026-08-30,sim,Cobertura potencial inferida por metadados no ...
4,P05,-15.542658,-47.979778,1166,2023-07-26,2026-08-30,sim,Cobertura potencial inferida por metadados no ...
5,P06,-15.543083,-47.979286,1166,2023-07-26,2026-08-30,sim,Cobertura potencial inferida por metadados no ...
6,P07,-15.541673,-47.979366,1166,2023-07-26,2026-08-30,sim,Cobertura potencial inferida por metadados no ...
7,P08,-15.537334,-47.975750,1166,2023-07-26,2026-08-30,sim,Cobertura potencial inferida por metadados no ...
8,P09,-15.537017,-47.975281,1166,2023-07-26,2026-08-30,sim,Cobertura potencial inferida por metadados no ...
9,P10,-15.536032,-47.976860,1166,2023-07-26,2026-08-30,sim,Cobertura potencial inferida por metadados no ...


## Limitações desta etapa

- A consulta usa metadados e bounding box; ainda não confirma se cada ponto está dentro da máscara de água ou do swath efetivo de cada granulo.
- Nenhum produto PIXC pesado é baixado.
- A contagem por ponto é preliminar e representa cobertura potencial do envelope do conjunto.
- A validação fina deve ser feita depois com uma passagem teste e/ou footprints mais detalhados.

## Interpretação da tabela final

A coluna `n_passagens_encontradas` indica quantos metadados/produtos SWOT foram encontrados para a área consultada. `primeira_data` e `ultima_data` delimitam o período observado no catálogo. `cobertura_potencial = sim` significa que há evidência catalográfica de cobertura SWOT para o envelope com buffer, mas não garante observação útil em cada exutório.